### ```__hash__```

In the following, ```C``` is a class and ```x``` and ```y``` are its instances. ```__hash__``` returns an ```int``` value, with the following requirement:

```x==y implies x.__hash__() == y.__hash__()```

The value is used in storing objects in dictionaries and sets. The instances ```x``` and ```y``` must be immutable

__hash__ — Makes Instances Usable as Dict Keys / Set Elements

Remember way back — "dictionary keys must be hashable"? __hash__ is exactly the mechanism that makes that work:

python
hash(x)     # actually calls   x.__hash__()

The rule stated: x == y implies x.__hash__() == y.__hash__() — meaning: if two objects are considered "equal", they MUST produce the same hash number. Otherwise, dictionaries and sets would get confused trying to look things up (recall the "fingerprint" analogy from your hashability lesson).

"instances must be immutable" — same requirement you already learned for tuples: if the object could change after being hashed, its "address" in the dictionary would become wrong.

## A Simple `__hash__` Example

Let's build a small class, watch the default behavior, then override `__hash__` and `__eq__` ourselves — and see exactly why the rule matters.

---

### Step 1 — Without Any Custom Methods (the default behavior)

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

p1 = Point(1, 2)
p2 = Point(1, 2)     # same x,y values, but a DIFFERENT object

print(p1 == p2)          # → False!   default == checks IDENTITY, not values
print(hash(p1) == hash(p2))   # → False   (usually — default hash is identity-based too)
```

By default, Python treats `p1` and `p2` as **completely unrelated** objects — even though they hold the same `x` and `y` — because without a custom `__eq__`, `==` just asks *"are these literally the same object in memory?"*

---

### Step 2 — Add `__eq__` So "Same Values" Means "Equal"

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

p1 = Point(1, 2)
p2 = Point(1, 2)

print(p1 == p2)     # → True    ✓ now compares VALUES
```

Great — `==` now behaves the way you probably want. **But** there's a hidden problem now...

---

### Step 3 — Try Using It as a Set Element / Dict Key

```python
points = {p1, p2}
```
```
TypeError: unhashable type: 'Point'
```

**Surprise!** The moment you define your own `__eq__`, Python **disables** the default `__hash__` automatically. Why? Because Python can't be sure your custom equality still respects the rule *"equal objects must hash the same"* — so it plays it safe and removes hashability entirely, forcing you to be explicit.

---

### Step 4 — Add `__hash__` to Fix It, Following the Rule

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __hash__(self):
        return hash((self.x, self.y))     # combine x,y into one hash, via a tuple!

p1 = Point(1, 2)
p2 = Point(1, 2)

print(p1 == p2)                       # → True
print(hash(p1) == hash(p2))            # → True   ✓ the RULE is satisfied!

points = {p1, p2}
print(len(points))                      # → 1     ✓ treated as duplicates, correctly!
```

Notice the trick: `hash((self.x, self.y))` — turn the object's defining values into a **tuple**, then hash the tuple (tuples are hashable, remember!). This guarantees: if two `Point`s have the same `x` and `y`, they'll **always** produce the same hash — exactly satisfying the rule *"x==y implies x.__hash__() == y.__hash__()"*.

---

### Why This Actually Matters — Watch It Work in a Set

```python
p1 = Point(1, 2)
p2 = Point(1, 2)   # a "duplicate" — same values, different object
p3 = Point(9, 9)

points = {p1, p2, p3}
print(len(points))     # → 2     ← p1 and p2 correctly COLLAPSED into one!
```

Without the correct `__hash__`/`__eq__` pairing, `p1` and `p2` would be treated as **two separate** entries — even though logically they represent "the same point." With them correctly defined, the set behaves the way you'd naturally expect.

---

### Why "Instances Must Be Immutable" Matters Here — A Broken Example

Watch what happens if you let the hashed values **change** after being used as a dict key:

```python
class MutablePoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    def __hash__(self):
        return hash((self.x, self.y))

mp = MutablePoint(1, 2)
d = {mp: "first point"}

mp.x = 99          # ⚠ MUTATING it AFTER it's already a dict key!

print(d.get(mp))     # → None!!  Lost! Even though 'mp' is literally the SAME object!
```

**What went wrong:** the dictionary stored `mp` under its **original** hash (based on `x=1, y=2`). After you changed `mp.x` to `99`, its hash **changed too** — but the dictionary's internal storage still expects the **old** hash. Now `d.get(mp)` computes the **new** hash, looks in the wrong spot, and finds nothing — the entry is effectively **lost**, even though the exact same object is sitting right there.

This is precisely why the rule insists: **only use immutable data as the basis for `__hash__`**, and don't design hashable classes whose hash-relevant fields can change after creation.

---

### Comparison Table — What Each Version Supports

| Version | `p1 == p2` (same values)? | Usable in a set/dict? |
|---|---|---|
| No `__eq__`, no `__hash__` | ✗ False (identity check) | ✓ yes (default hash, by identity) |
| Custom `__eq__` only | ✓ True | ✗ TypeError — unhashable! |
| Custom `__eq__` + `__hash__` | ✓ True | ✓ yes — behaves correctly |

---

### The One-Sentence Summary

> Defining a custom `__eq__` makes "equal values" mean equal, but Python then **disables** the default hash to protect you — you must also define `__hash__` (typically `hash((field1, field2, ...))`) so that equal objects reliably produce the same hash number, which is exactly what lets sets and dictionaries correctly recognize "logical duplicates" instead of treating every instance as unique. And if the hashed fields can change after the object is already stored as a key, the dictionary loses track of it — which is why hashable objects should be effectively immutable. 🎯

## What "The Dictionary Still Expects the Old Hash" Means

Let's open up how a dictionary **physically organizes** its data — this is the mechanism the sentence is describing, and it directly connects to your namespace lesson (*"the box is literally implemented as a dictionary"*), just one level deeper.

---

### The Dictionary Isn't a List — It's Organized by Hash "Address"

You've been treating dictionaries as magic boxes: *"put key → value in, get value back out."* Under the hood, Python uses the **hash number** to decide **exactly where** in memory to physically place that entry — like a filing system with numbered slots.

```
d = {mp: "first point"}
```

When this line runs, Python does roughly:

```
1. Compute hash(mp)   →  let's say this gives:  hash = 5000
2. Use 5000 to pick a STORAGE SLOT — say, slot #12 (some internal calculation)
3. Physically place ("first point") into slot #12
```

**The slot number is derived from the hash at the moment of insertion.** Once placed, the dictionary doesn't re-check or re-file it later — it just trusts that slot #12 is where that entry lives, based on the hash it had **at insertion time**.

---

### Now You Mutate `mp`

```python
mp.x = 99
```

The **object** `mp` still exists, same memory address, same identity. But its **hash value changes**:

```
hash(mp) BEFORE:  5000   (based on x=1, y=2)
hash(mp) AFTER:   9999   (based on x=99, y=2)    ← DIFFERENT number now!
```

**Nothing tells the dictionary this happened.** The dictionary has no way of knowing you changed `mp.x` — it's not watching the object; it just has "first point" sitting in slot #12, filed under the assumption that anything hashing to 5000 belongs there.

---

### The Lookup — `d.get(mp)` — Looks in the WRONG Slot

```python
d.get(mp)
```

```
1. Compute hash(mp) RIGHT NOW  →  9999   (the NEW hash, since x is now 99)
2. Use 9999 to calculate a storage slot  →  say, slot #47
3. Look in slot #47  →  ... it's EMPTY!  (nothing was ever placed there)
4. Return None
```

**The word "still expects the old hash"** means: the entry `"first point"` is physically sitting in **slot #12** (where it was filed based on the hash **at insertion time**, 5000). But your lookup, using the object's **current** hash (9999), searches **slot #47** instead. Two different slots — the data and the search never meet.

---

### Visualizing It

```
INSERTION TIME (hash=5000):

  dictionary storage:
  slot 12: [mp → "first point"]     ← filed HERE, based on hash 5000
  slot 47: [ empty ]


AFTER mp.x = 99  (hash is NOW 9999, but nothing moved!):

  dictionary storage:
  slot 12: [mp → "first point"]     ← STILL here — untouched, unaware of the mutation!
  slot 47: [ empty ]


LOOKUP d.get(mp):

  compute hash(mp) → 9999 (current)
  look in slot 47 (where 9999 points)
  → slot 47 is EMPTY
  → return None
```

The actual data never moved — it's genuinely still sitting in slot 12. But your lookup, using the **new** hash, is searching in slot 47. **Mismatch = the entry appears lost, even though it's technically still there, unreachable.**

---

### An Analogy — A Library With a Broken Card Catalog

> Imagine a library where books are shelved by a **code number computed from the book's title**.
>
> - You shelve *"Point (1,2)"* → code 5000 → **shelf 12**.
> - Later, someone **silently changes the book's title** (crosses it out, writes a new one) — but nobody re-shelves the book. It's still physically sitting on shelf 12.
> - Now you want to find it. You compute the code from its **current (new) title** → get 9999 → walk to **shelf 47**.
> - Shelf 47 is empty. You conclude "the book is missing" — even though it's literally still on shelf 12, just filed under a code that no longer matches where you're looking.

The book isn't lost from the *building* — it's lost from **your ability to find it**, because your search method (recompute the code from the current title) no longer matches where it was actually placed (the code from the *original* title).

---

### Confirming With Code

```python
print(list(d.keys())[0] is mp)      # → True!  It's LITERALLY the same object, still in there
print(d.get(mp))                      # → None   but lookup FAILS
```

**Proof:** if you iterate the dictionary directly, `mp` (or something `==` to it) really is present. It's only the **hash-based lookup** — `d[mp]` or `d.get(mp)` — that fails, because that specific operation trusts the hash, and the hash has drifted away from what was used at insertion.

---

### The One-Sentence Summary

> A dictionary uses an object's hash value to decide **which physical slot** to store an entry in — and it makes this decision **once, at insertion time**, then never revisits it. If the object later mutates and its hash changes, the dictionary's stored entry doesn't move to match — it stays in its original slot. So `d.get(mp)` computes the object's **new** hash, searches the **new, wrong slot**, finds nothing there, and returns `None` — even though the entry is still physically present, just filed under a slot number that no longer matches what a fresh hash computation points to. 🎯